# Notebook 2 — Consolidação de Trials + Treino Final + Grad-CAM

Carrega os JSONs de todas as máquinas, escolhe o melhor trial por F1 macro,
faz o treino final com CV 3-fold e gera análise Grad-CAM.

## 1. Imports e configuração

In [ ]:
import os, json, glob, random, shutil
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
from PIL import Image

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    f1_score as sk_f1,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
)

import wandb
import matplotlib.pyplot as plt

# Reprodutibilidade
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## 2. Configuração

In [ ]:
# ── Pasta com todos os JSONs de todas as máquinas ────────────────────────────
# Coloca todos os JSONs de todas as máquinas numa pasta só antes de rodar
TRIALS_DIR    = "/kaggle/input/datasets/jniorviana/todastrials-3estudos/todastrials"   # ajusta para o path do dataset

# ── Treino final ──────────────────────────────────────────────────────────────
N_FOLDS       = 3
VAL_SPLIT     = 0.10
EPOCHS_FINAL  = 30
BATCH_SIZE    = 64
WANDB_PROJECT = "fer-anynet-final-30"

# ── Dataset ───────────────────────────────────────────────────────────────────
DATASET_DIR = "/kaggle/input/datasets/arnabkumarroy02/ferplus"

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_DIR = "output_final"
os.makedirs(OUTPUT_DIR, exist_ok=True)

from kaggle_secrets import UserSecretsClient
os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
wandb.login()


## 3. Carrega todos os JSONs e seleciona o melhor trial

In [ ]:
jsons = sorted(glob.glob(os.path.join(TRIALS_DIR, "**/*.json"), recursive=True))
print(f"Total de JSONs encontrados: {len(jsons)}")

trials = []
for path in jsons:
    with open(path) as f:
        data = json.load(f)
    trials.append(data)

# Ordena por mean_val_f1 decrescente
trials.sort(key=lambda x: x["mean_val_f1"], reverse=True)

print(f"\n{'Rank':<5} {'Trial ID':<25} {'mean_f1':<10} {'std_f1':<10} {'F1 por fold'}")
print("-" * 80)
for i, t in enumerate(trials[:10], start=1):   # top 10
    folds = [f"{f:.4f}" for f in t["val_f1_folds"]]
    print(f"{i:<5} {t['trial_id']:<25} {t['mean_val_f1']:<10.4f} {t['std_val_f1']:<10.4f} {folds}")

best_trial = trials[0]
print(f"\n{'='*80}")
print(f"Melhor trial  : {best_trial['trial_id']}")
print(f"mean_val_f1   : {best_trial['mean_val_f1']:.4f} ± {best_trial['std_val_f1']:.4f}")
print(f"Config        : {best_trial['config']}")
print(f"Hparams       : {best_trial['hparams']}")


## 4. Dataset

In [ ]:
transform_train = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((48, 48)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3),
])

transform_eval = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3),
])

all_samples, all_labels = [], []
for split in ["train", "validation", "test"]:
    folder = datasets.ImageFolder(os.path.join(DATASET_DIR, split))
    all_samples += folder.samples
    all_labels  += folder.targets

all_samples = np.array(all_samples, dtype=object)
all_labels  = np.array(all_labels,  dtype=int)
classes     = folder.classes
num_classes = len(classes)

print(f"Total de imagens : {len(all_samples)}")
print(f"Classes          : {classes}")


class FERPlusDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label


## 5. Arquitetura AnyNet

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, ch, ratio=4):
        super().__init__()
        r = max(1, ch // ratio)
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(ch, r, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(r, ch, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        w = self.se(x).view(x.size(0), -1, 1, 1)
        return x * w


class PlainConv(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, **kwargs):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x):
        return self.conv(x) + self.shortcut(x)


class MBConv(nn.Module):
    def __init__(self, in_ch, out_ch, expand_ratio=6, kernel_size=3, stride=1, **kwargs):
        super().__init__()
        mid_ch  = in_ch * expand_ratio
        padding = kernel_size // 2
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, mid_ch, 1, bias=False),
            nn.BatchNorm2d(mid_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_ch, mid_ch, kernel_size, stride=stride,
                      padding=padding, groups=mid_ch, bias=False),
            nn.BatchNorm2d(mid_ch),
            nn.ReLU(inplace=True),
            SEBlock(mid_ch),
            nn.Conv2d(mid_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
        )
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x):
        return self.conv(x) + self.shortcut(x)


class Stem(nn.Module):
    def __init__(self, out_ch=32):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, out_ch, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.stem(x)


class AnyNetStage(nn.Module):
    def __init__(self, in_ch, out_ch, depth, block_type):
        super().__init__()
        blocks = []
        for i in range(depth):
            stride = 2 if i == 0 else 1
            blocks.append(block_type(in_ch if i == 0 else out_ch, out_ch, stride=stride))
        self.blocks = nn.Sequential(*blocks)

    def forward(self, x):
        return self.blocks(x)


class AnyNet(nn.Module):
    def __init__(self, num_stages, widths, depths, transition_stage,
                 dropout, num_classes=8):
        super().__init__()
        self.stem = Stem(out_ch=32)
        stages, in_ch = [], 32
        for i in range(num_stages):
            block_type = MBConv if i >= transition_stage - 1 else PlainConv
            stages.append(AnyNetStage(in_ch, widths[i], depths[i], block_type))
            in_ch = widths[i]
        self.stages = nn.Sequential(*stages)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(widths[-1], num_classes),
        )
        # Registra camada alvo para Grad-CAM
        last_stage = list(self.stages.children())[-1]
        last_block = list(last_stage.blocks.children())[-1]
        self.gradcam_target = last_block.conv[-1]

    def forward(self, x):
        x = self.stem(x)
        x = self.stages(x)
        return self.head(x)


def build_anynet(config, num_classes=8) -> AnyNet:
    return AnyNet(
        num_stages       = config["num_stages"],
        widths           = config["widths"],
        depths           = config["depths"],
        transition_stage = config["transition_stage"],
        dropout          = config["dropout"],
        num_classes      = num_classes,
    )


## 6. Funções de treino e avaliação

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        correct      += (outputs.argmax(1) == labels).sum().item()
        total        += labels.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device, return_preds=False):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs      = model(imgs)
        loss         = criterion(outputs, labels)
        preds        = outputs.argmax(1)
        running_loss += loss.item() * imgs.size(0)
        correct      += (preds == labels).sum().item()
        total        += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    avg_loss   = running_loss / total
    acc        = correct / total
    f1_macro   = sk_f1(all_labels, all_preds, average="macro", zero_division=0)
    precision_macro = precision_score(
        all_labels, all_preds, average="macro", zero_division=0
    )
    recall_macro = recall_score(
        all_labels, all_preds, average="macro", zero_division=0
    )

    if return_preds:
        return (
            avg_loss,
            acc,
            f1_macro,
            precision_macro,
            recall_macro,
            all_preds,
            all_labels,
        )
    return avg_loss, acc, f1_macro, precision_macro, recall_macro


## 7. Treino final com CV 3-fold

In [ ]:
skf        = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
cv_results = {}

for fold_idx, (trainval_idx, test_idx) in enumerate(
    skf.split(all_samples, all_labels), start=1
):
    print(f'\n{"="*60}')
    print(f'TREINO FINAL — FOLD {fold_idx}/{N_FOLDS}')

    train_idx, val_idx = train_test_split(
        trainval_idx,
        test_size=VAL_SPLIT,
        random_state=SEED,
        stratify=all_labels[trainval_idx],
    )

    print(f'  Treino     : {len(train_idx)}')
    print(f'  Val (~10%) : {len(val_idx)}')
    print(f'  Teste      : {len(test_idx)}')

    train_loader_fold = DataLoader(
        FERPlusDataset(all_samples[train_idx].tolist(), transform=transform_train),
        batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True,
    )
    val_loader_fold = DataLoader(
        FERPlusDataset(all_samples[val_idx].tolist(), transform=transform_eval),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True,
    )
    test_loader_fold = DataLoader(
        FERPlusDataset(all_samples[test_idx].tolist(), transform=transform_eval),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True,
    )

    run = wandb.init(
        project=WANDB_PROJECT,
        group="final_training",
        name=f"final_fold_{fold_idx}",
        config={
            **best_trial["config"],
            "lr"          : best_trial["hparams"]["lr"],
            "weight_decay": best_trial["hparams"]["weight_decay"],
            "epochs"      : EPOCHS_FINAL,
            "trial_id"    : best_trial["trial_id"],
        },
        reinit=True,
    )

    model     = build_anynet(best_trial["config"], num_classes=num_classes).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(
        model.parameters(),
        lr=best_trial["hparams"]["lr"],
        weight_decay=best_trial["hparams"]["weight_decay"],
    )
    scheduler          = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_FINAL)
    best_val_f1        = 0.0
    best_val_loss      = None
    best_val_precision = 0.0
    best_val_recall    = 0.0
    best_state         = None

    for epoch in range(EPOCHS_FINAL):
        train_loss, train_acc = train_one_epoch(
            model, train_loader_fold, criterion, optimizer, device
        )
        val_loss, val_acc, val_f1, val_precision, val_recall = evaluate(
            model, val_loader_fold, criterion, device
        )
        scheduler.step()

        wandb.log({
            "epoch"        : epoch + 1,
            "train_loss"   : train_loss,
            "train_acc"    : train_acc,
            "val_loss"     : val_loss,
            "val_acc"      : val_acc,
            "val_f1"       : val_f1,
            "val_precision": val_precision,
            "val_recall"   : val_recall,
        })

        print(f"  Epoch {epoch+1:02d}/{EPOCHS_FINAL} | "
              f"train {train_loss:.4f}/{train_acc:.4f} | "
              f"val_loss {val_loss:.4f} | "
              f"val_f1 {val_f1:.4f} | "
              f"val_precision {val_precision:.4f} | "
              f"val_recall {val_recall:.4f}")

        if val_f1 > best_val_f1:
            best_val_f1        = val_f1
            best_val_loss      = val_loss
            best_val_precision = val_precision
            best_val_recall    = val_recall
            best_state         = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)

    # ── Salva modelo ──────────────────────────────────────────────────────
    model_path = os.path.join(OUTPUT_DIR, f"anynet_fold_{fold_idx}.pth")
    torch.save({
        "model_state_dict"   : best_state,
        "config"             : best_trial["config"],
        "hparams"            : best_trial["hparams"],
        "trial_id"           : best_trial["trial_id"],
        "fold"               : fold_idx,
        "best_val_f1"        : best_val_f1,
        "best_val_loss"      : best_val_loss,
        "best_val_precision" : best_val_precision,
        "best_val_recall"    : best_val_recall,
        "classes"            : classes,
    }, model_path)
    print(f"  Modelo salvo: {model_path}")

    # ── Teste do fold ─────────────────────────────────────────────────────
    criterion_test = nn.CrossEntropyLoss()
    (
        test_loss,
        fold_test_acc,
        fold_test_f1,
        fold_test_precision,
        fold_test_recall,
        preds,
        labels,
    ) = evaluate(model, test_loader_fold, criterion_test, device, return_preds=True)

    report_text = classification_report(
        labels, preds, target_names=classes, zero_division=0
    )
    report_dict = classification_report(
        labels, preds, target_names=classes, zero_division=0, output_dict=True
    )

    print(
        f'\n  TESTE fold {fold_idx} — '
        f'Loss={test_loss:.4f} | '
        f'Acc={fold_test_acc:.4f} | '
        f'F1={fold_test_f1:.4f} | '
        f'Precision={fold_test_precision:.4f} | '
        f'Recall={fold_test_recall:.4f}'
    )
    print(report_text)

    report_txt_path = os.path.join(OUTPUT_DIR, f"classification_report_fold_{fold_idx}.txt")
    report_json_path = os.path.join(OUTPUT_DIR, f"classification_report_fold_{fold_idx}.json")
    with open(report_txt_path, "w") as f:
        f.write(report_text)
    with open(report_json_path, "w") as f:
        json.dump(report_dict, f, indent=2)

    wandb.summary["best_val_f1"]        = best_val_f1
    wandb.summary["best_val_loss"]      = best_val_loss
    wandb.summary["best_val_precision"] = best_val_precision
    wandb.summary["best_val_recall"]    = best_val_recall
    wandb.summary["test_loss"]          = test_loss
    wandb.summary["test_acc"]           = fold_test_acc
    wandb.summary["test_f1"]            = fold_test_f1
    wandb.summary["test_precision"]     = fold_test_precision
    wandb.summary["test_recall"]        = fold_test_recall
    run.finish()

    cv_results[fold_idx] = {
        "test_loss"       : test_loss,
        "test_acc"        : fold_test_acc,
        "test_f1"         : fold_test_f1,
        "test_precision"  : fold_test_precision,
        "test_recall"     : fold_test_recall,
        "classification_report": report_dict,
        "cm"              : confusion_matrix(labels, preds),
        "preds"           : preds,
        "labels"          : labels,
        "model_path"      : model_path,
        "report_txt_path" : report_txt_path,
        "report_json_path": report_json_path,
        "model"           : model,
    }

    torch.cuda.empty_cache()

# ── Resumo final ───────────────────────────────────────────────────────────────
accs        = [cv_results[f]["test_acc"]       for f in cv_results]
f1s         = [cv_results[f]["test_f1"]        for f in cv_results]
precisions  = [cv_results[f]["test_precision"] for f in cv_results]
recalls     = [cv_results[f]["test_recall"]    for f in cv_results]
losses      = [cv_results[f]["test_loss"]      for f in cv_results]

summary = {
    "fold_1_acc": accs[0], "fold_2_acc": accs[1], "fold_3_acc": accs[2],
    "fold_1_f1" : f1s[0],  "fold_2_f1" : f1s[1],  "fold_3_f1" : f1s[2],
    "fold_1_precision": precisions[0], "fold_2_precision": precisions[1], "fold_3_precision": precisions[2],
    "fold_1_recall"   : recalls[0],    "fold_2_recall"   : recalls[1],    "fold_3_recall"   : recalls[2],
    "fold_1_loss"     : losses[0],     "fold_2_loss"     : losses[1],     "fold_3_loss"     : losses[2],
    "mean_acc"       : float(np.mean(accs)),       "std_acc"       : float(np.std(accs)),
    "mean_f1"        : float(np.mean(f1s)),        "std_f1"        : float(np.std(f1s)),
    "mean_precision" : float(np.mean(precisions)), "std_precision" : float(np.std(precisions)),
    "mean_recall"    : float(np.mean(recalls)),    "std_recall"    : float(np.std(recalls)),
    "mean_loss"      : float(np.mean(losses)),     "std_loss"      : float(np.std(losses)),
}

summary_path = os.path.join(OUTPUT_DIR, "cv_metrics_summary.json")
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

run = wandb.init(project=WANDB_PROJECT, name="cv_summary", reinit=True)
wandb.log(summary)
wandb.summary["summary_path"] = summary_path
run.finish()

print(f'\n{"="*60}')
print(f'Loss por fold      : {[f"{x:.4f}" for x in losses]}')
print(f'Acc por fold       : {[f"{a:.4f}" for a in accs]}')
print(f'F1 por fold        : {[f"{f:.4f}" for f in f1s]}')
print(f'Precision por fold : {[f"{p:.4f}" for p in precisions]}')
print(f'Recall por fold    : {[f"{r:.4f}" for r in recalls]}')
print(f'Loss — Média       : {np.mean(losses):.4f} ± {np.std(losses):.4f}')
print(f'Acc — Média        : {np.mean(accs):.4f} ± {np.std(accs):.4f}')
print(f'F1 — Média         : {np.mean(f1s):.4f} ± {np.std(f1s):.4f}')
print(f'Precision — Média  : {np.mean(precisions):.4f} ± {np.std(precisions):.4f}')
print(f'Recall — Média     : {np.mean(recalls):.4f} ± {np.std(recalls):.4f}')
print(f'Resumo salvo em    : {summary_path}')


## 8. Matriz de confusão por fold

In [ ]:
import seaborn as sns

fig, axes = plt.subplots(1, N_FOLDS, figsize=(18, 5))
for fold_idx, ax in enumerate(axes, start=1):
    cm = cv_results[fold_idx]["cm"]
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=classes, yticklabels=classes, ax=ax)
    ax.set_title(f"Fold {fold_idx} — F1={cv_results[fold_idx]['test_f1']:.4f}")
    ax.set_xlabel("Predito")
    ax.set_ylabel("Real")
    ax.tick_params(axis="x", rotation=45)

plt.suptitle("Matrizes de Confusão — AnyNet FER+", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrices.png"), dpi=150, bbox_inches="tight")
plt.show()


## 9. Grad-CAM

In [ ]:
import torch.nn.functional as F

class GradCAM:
    def __init__(self, model):
        self.model       = model
        self.activations = None
        self.gradients   = None
        self.target_layer = model.gradcam_target
        self.target_layer.register_forward_hook(self._save_activation)
        self.target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, _, __, output):
        self.activations = output.detach()

    def _save_gradient(self, _, __, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, img_tensor, class_idx):
        self.model.eval()
        img_tensor = img_tensor.unsqueeze(0).to(device)
        self.model.zero_grad()
        logits = self.model(img_tensor)
        logits[0, class_idx].backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam     = (weights * self.activations).sum(dim=1, keepdim=True)
        cam     = F.relu(cam)
        cam     = F.interpolate(cam, size=(48, 48), mode="bilinear", align_corners=False)
        cam     = cam.squeeze().cpu().numpy()
        cam     = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam


def tensor_to_img(tensor):
    img = tensor.permute(1, 2, 0).cpu().numpy()
    img = img * 0.5 + 0.5
    return np.clip(img, 0, 1)


def overlay_cam(img_np, cam):
    heatmap = plt.cm.jet(cam)[..., :3]
    return np.clip(0.5 * img_np + 0.5 * heatmap, 0, 1)


@torch.no_grad()
def collect_predictions(model, loader):
    model.eval()
    all_imgs, all_labels, all_preds = [], [], []
    for imgs, labels in loader:
        outputs = model(imgs.to(device))
        preds   = outputs.argmax(1).cpu()
        all_imgs.append(imgs)
        all_labels.append(labels)
        all_preds.append(preds)
    return torch.cat(all_imgs), torch.cat(all_labels).numpy(), torch.cat(all_preds).numpy()


def gradcam_analysis(model, classes, loader=None, image_dir=None, target_class=None, fold_idx=1):
    assert loader is not None or image_dir is not None, "Forneça loader ou image_dir"

    if loader is not None:
        imgs, true_labels, pred_labels = collect_predictions(model, loader)
    else:
        imgs_list, labels_list = [], []
        for class_idx, class_name in enumerate(classes):
            class_path = os.path.join(image_dir, class_name)
            if not os.path.exists(class_path):
                continue
            files = [f for f in os.listdir(class_path)
                     if f.lower().endswith((".png", ".jpg", ".jpeg"))]
            for fname in files:
                img = Image.open(os.path.join(class_path, fname)).convert("RGB")
                imgs_list.append(transform_eval(img))
                labels_list.append(class_idx)
        imgs        = torch.stack(imgs_list)
        true_labels = np.array(labels_list)
        with torch.no_grad():
            model.eval()
            pred_labels = model(imgs.to(device)).argmax(1).cpu().numpy()

    if target_class is None:
        target_class = random.randint(0, len(classes) - 1)
    class_name = classes[target_class]
    print(f"Classe alvo: {class_name} (idx={target_class})")

    is_target_true = (true_labels == target_class)
    is_target_pred = (pred_labels == target_class)

    categories = {
        "VP (era, acertou)"   : np.where( is_target_true &  is_target_pred)[0],
        "FP (não era, disse)" : np.where(~is_target_true &  is_target_pred)[0],
        "FN (era, não disse)" : np.where( is_target_true & ~is_target_pred)[0],
        "VN (não era, ok)"    : np.where(~is_target_true & ~is_target_pred)[0],
    }

    for cat, idx in categories.items():
        print(f"  {cat}: {len(idx)} disponíveis")

    gradcam = GradCAM(model)
    colors  = {"VP (era, acertou)": "green", "FP (não era, disse)": "red",
                "FN (era, não disse)": "orange", "VN (não era, ok)": "blue"}

    fig, axes = plt.subplots(nrows=4, ncols=2, figsize=(6, 12))
    fig.suptitle(f"Grad-CAM — Classe: {class_name} | Fold {fold_idx}", fontsize=13, fontweight="bold")

    for row, (cat_name, idx_arr) in enumerate(categories.items()):
        for col in range(2):
            ax = axes[row, col]
            if len(idx_arr) == 0:
                ax.text(0.5, 0.5, "Sem exemplos", ha="center", va="center")
                ax.axis("off")
                continue
            sample_idx = np.random.choice(idx_arr)
            img_tensor = imgs[sample_idx]
            true_label = true_labels[sample_idx]
            pred_label = pred_labels[sample_idx]
            cam        = gradcam.generate(img_tensor, class_idx=int(pred_label))
            overlay    = overlay_cam(tensor_to_img(img_tensor), cam)
            ax.imshow(overlay)
            ax.set_title(f"Real: {classes[true_label]}\nPred: {classes[pred_label]}",
                         fontsize=8, color=colors[cat_name])
            ax.axis("off")
        axes[row, 0].set_ylabel(cat_name, fontsize=8, rotation=90, labelpad=40)

    plt.tight_layout()
    save_path = os.path.join(OUTPUT_DIR, f"gradcam_{class_name}_fold{fold_idx}.png")
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Salvo: {save_path}")


## 10. Execução do Grad-CAM

In [ ]:
# Usa o modelo do fold 1 — pode trocar para fold_idx=2 ou 3
fold_idx   = 1
model_gcam = cv_results[fold_idx]["model"].to(device)

# Monta loader de teste do fold 1
_, (trainval_idx, test_idx) = next(enumerate(skf.split(all_samples, all_labels)))
test_loader_gcam = DataLoader(
    FERPlusDataset(all_samples[test_idx].tolist(), transform=transform_eval),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True,
)

# Modo dataset — sorteia uma classe aleatória
gradcam_analysis(
    model       = model_gcam,
    classes     = classes,
    loader      = test_loader_gcam,
    target_class= None,   # None = sorteia
    fold_idx    = fold_idx,
)

# Modo externo — pasta com subpastas por classe
# gradcam_analysis(
#     model       = model_gcam,
#     classes     = classes,
#     image_dir   = "/caminho/outro/dataset",
#     target_class= 4,
#     fold_idx    = fold_idx,
# )
